# CASMI 2026 — spectral library retrieval baseline

Baseline for the Enveda CASMI26 molecule-ID competition. **Covers novelty class 1 only** (structures with public reference spectra in `train.parquet`); molecules outside the library will score 0 here.

Pipeline:
1. Convert each test molecule's spectra to a neutral mass (via adduct) and shortlist training **structures** (`inchikey14`) whose formula mass matches within a tolerance.
2. Load only the shortlisted training spectra, bin them (sqrt intensity, L2-normalised sparse vectors).
3. Cosine-score every test spectrum against every shortlisted library spectrum; keep the best match per (test spectrum, structure).
4. Aggregate across a molecule's spectra (mean of per-spectrum best scores) → rank structures → top 25 SMILES joined by `;`.

Includes a local validation (hold out spectra of known library molecules) so you can iterate without leaderboard submissions. Note the visible `test.parquet` is sampled from train, so scoring it here is leaky — use the validation section.

Next steps: neutral-loss features / modified cosine, spectrum->fingerprint model + PubChem/COCONUT candidates (class 2), de novo (class 3).

In [ ]:
import os, re, time, glob
import numpy as np, pandas as pd, pyarrow.parquet as pq, pyarrow.compute as pc, pyarrow as pa
import scipy.sparse as sp

CFG = dict(
    bin_width=0.02, max_mz=1300.0, min_rel_int=0.01, max_peaks=100, min_peaks=2,
    mass_tol_da=0.01, mass_tol_ppm=15.0, fallback_tol_da=0.5, topk=25,
    max_cands_per_mol=20000,
)
DATA_DIR = os.environ.get("CASMI_DIR") or next(iter(glob.glob("/kaggle/input/*/train.parquet")), None)
DATA_DIR = os.path.dirname(DATA_DIR) if DATA_DIR and DATA_DIR.endswith(".parquet") else (DATA_DIR or "data")
OUT_DIR = os.environ.get("CASMI_OUT", "/kaggle/working" if os.path.isdir("/kaggle/working") else ".")
print(DATA_DIR, OUT_DIR)

In [ ]:
# ---- masses ----
EL = {"H":1.00782503207,"C":12.0,"N":14.0030740048,"O":15.99491461956,"F":18.99840322,"Na":22.9897692809,
      "P":30.97376163,"S":31.972071,"Cl":34.96885268,"K":38.96370668,"Br":78.9183371,"I":126.904473,
      "B":11.0093054,"Si":27.9769265325,"Se":79.9165213,"As":74.9215965,"Fe":55.9349375,"Mg":23.985041699,
      "Ca":39.96259098,"Zn":63.9291422,"Cu":62.9295975,"Li":7.016003437,"Pt":194.964791,"Sn":119.9022016,
      "Hg":201.970643,"Co":58.933195,"Mn":54.9380451,"Ni":57.9353429,"Al":26.98153863,"Ge":73.9211778,"Sb":120.903818}
_tok = re.compile(r"([A-Z][a-z]?)(\d*)")
def formula_mass(f):
    if not isinstance(f, str) or not f: return np.nan
    m = 0.0
    for el, n in _tok.findall(f):
        if el not in EL: return np.nan
        m += EL[el] * (int(n) if n else 1)
    return m

PROTON, E = 1.007276467, 0.000548580
H2O, NH4, NA, K, CL, CH2O2 = 18.0105646863, 18.0343740, 22.9897692809, 38.96370668, 34.96885268, 46.00547930326
# neutral M = precursor_mz - ADDUCT_DELTA
ADDUCT_DELTA = {
    "[M+H]+": PROTON, "[M+NH4]+": NH4 + 0 - E, "[M-H2O+H]+": PROTON - H2O, "[M-2H2O+H]+": PROTON - 2*H2O,
    "[M+Na]+": NA - E, "[M+K]+": K - E, "[M-H]-": -PROTON, "[M-H2O-H]-": -PROTON - H2O,
    "[M+CH2O2-H]-": CH2O2 - PROTON, "[M+Cl]-": CL + E,
}
def neutral_mass(df):
    d = df["adduct"].map(ADDUCT_DELTA)
    return (df["precursor_mz"] - d).to_numpy(float)

In [ ]:
# ---- spectrum -> sparse vector ----
NB = int(CFG["max_mz"] / CFG["bin_width"]) + 1
def vectorize(mzs_list, ints_list, prec):
    rows, cols, vals = [], [], []
    keep = np.zeros(len(mzs_list), bool)
    for i, (mz, it, p) in enumerate(zip(mzs_list, ints_list, prec)):
        mz = np.asarray(mz, float); it = np.asarray(it, float)
        if len(mz) == 0: continue
        it = it / it.max()
        m = (mz <= p + 2.0) & (it >= CFG["min_rel_int"]) & (mz < CFG["max_mz"]) & (mz > 0)
        mz, it = mz[m], it[m]
        if len(mz) < CFG["min_peaks"]: continue
        if len(mz) > CFG["max_peaks"]:
            top = np.argsort(-it)[:CFG["max_peaks"]]; mz, it = mz[top], it[top]
        b = (mz / CFG["bin_width"]).astype(np.int64)
        v = np.sqrt(it)
        s = pd.Series(v).groupby(b).max()          # merge peaks sharing a bin
        v = s.to_numpy(); b = s.index.to_numpy()
        v = v / np.linalg.norm(v)
        rows.append(np.full(len(b), i)); cols.append(b); vals.append(v); keep[i] = True
    if not rows: return sp.csr_matrix((len(mzs_list), NB)), keep
    X = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(len(mzs_list), NB))
    return X, keep

In [ ]:
# ---- library metadata (cheap columns only) ----
t0 = time.time()
LIB_META_COLS = ["spectrum_id", "inchikey14", "normalized_smiles", "molecular_formula"]
lib_meta = pq.read_table(f"{DATA_DIR}/train.parquet", columns=LIB_META_COLS).to_pandas()
print(len(lib_meta), "library spectra", lib_meta.inchikey14.nunique(), "structures", f"{time.time()-t0:.0f}s")

struct = (lib_meta.drop_duplicates("inchikey14")[["inchikey14", "normalized_smiles", "molecular_formula"]]
          .reset_index(drop=True))
struct["mass"] = struct.molecular_formula.map(formula_mass)
struct = struct.dropna(subset=["mass"]).sort_values("mass").reset_index(drop=True)
struct_mass = struct["mass"].to_numpy()
print(len(struct), "structures with computable mass")

In [ ]:
def mol_masses(test_df):
    """Per-molecule neutral mass = median over its spectra."""
    d = test_df.assign(nm=neutral_mass(test_df))
    return d.groupby("molecule_id")["nm"].median()

def shortlist(mass):
    tol = max(CFG["mass_tol_da"], mass * CFG["mass_tol_ppm"] * 1e-6)
    lo, hi = np.searchsorted(struct_mass, [mass - tol, mass + tol])
    if hi - lo == 0:                       # widen if nothing matches (e.g. unknown adduct)
        lo, hi = np.searchsorted(struct_mass, [mass - CFG["fallback_tol_da"], mass + CFG["fallback_tol_da"]])
    if hi - lo > CFG["max_cands_per_mol"]:
        mid = (lo + hi) // 2; lo, hi = mid - CFG["max_cands_per_mol"]//2, mid + CFG["max_cands_per_mol"]//2
    return struct.inchikey14.iloc[lo:hi].tolist()

def load_library_vectors(keys, exclude_spectra=frozenset()):
    """Stream train.parquet and vectorize only spectra of the shortlisted structures."""
    keys = pa.array(sorted(set(keys)))
    pf = pq.ParquetFile(f"{DATA_DIR}/train.parquet")
    cols = ["spectrum_id", "inchikey14", "ms2_mzs", "ms2_normalized_intensities", "precursor_mz"]
    mats, metas = [], []
    for b in pf.iter_batches(batch_size=50000, columns=cols):
        t = pa.Table.from_batches([b])
        t = t.filter(pc.is_in(t["inchikey14"], value_set=keys))
        if t.num_rows == 0: continue
        df = t.to_pandas()
        if exclude_spectra: df = df[~df.spectrum_id.isin(exclude_spectra)]
        X, keep = vectorize(df.ms2_mzs, df.ms2_normalized_intensities, df.precursor_mz.to_numpy(float))
        mats.append(X[np.flatnonzero(keep)]); metas.append(df.loc[keep, ["spectrum_id", "inchikey14"]])
    X = sp.vstack(mats).tocsr() if mats else sp.csr_matrix((0, NB))
    meta = pd.concat(metas, ignore_index=True) if metas else pd.DataFrame(columns=["spectrum_id", "inchikey14"])
    return X, meta

In [ ]:
def predict(test_df, exclude_spectra=frozenset(), verbose=True):
    """Return {molecule_id: [(inchikey14, score), ...]} ranked, top-k."""
    mm = mol_masses(test_df)
    cand = {m: shortlist(x) for m, x in mm.items() if np.isfinite(x)}
    allkeys = set(k for v in cand.values() for k in v)
    t0 = time.time()
    LX, Lmeta = load_library_vectors(allkeys, exclude_spectra)
    if verbose: print(f"library: {LX.shape[0]} spectra for {len(allkeys)} shortlisted structures ({time.time()-t0:.0f}s)")
    lib_idx_by_key = Lmeta.groupby("inchikey14").indices
    out = {}
    for mol, g in test_df.groupby("molecule_id"):
        keys = [k for k in cand.get(mol, []) if k in lib_idx_by_key]
        if not keys: out[mol] = []; continue
        QX, keep = vectorize(g.ms2_mzs, g.ms2_normalized_intensities, g.precursor_mz.to_numpy(float))
        QX = QX[np.flatnonzero(keep)]
        if QX.shape[0] == 0: out[mol] = []; continue
        idx = np.concatenate([lib_idx_by_key[k] for k in keys])
        owner = np.concatenate([[j] * len(lib_idx_by_key[k]) for j, k in enumerate(keys)])
        S = (QX @ LX[idx].T).toarray()                         # n_query x n_lib_spectra
        best = np.zeros((S.shape[0], len(keys)))
        for j in range(len(keys)):                              # max over each structure's spectra
            best[:, j] = S[:, owner == j].max(axis=1)
        score = best.mean(axis=0)                               # mean over the molecule's spectra
        order = np.argsort(-score)[:CFG["topk"]]
        out[mol] = [(keys[j], float(score[j])) for j in order]
    return out

def to_submission(test_df, ranked, fallback="CCO"):
    smi = struct.set_index("inchikey14")["normalized_smiles"]
    rows = []
    for mol in test_df.molecule_id.unique():
        s = [smi[k] for k, _ in ranked.get(mol, [])] or [fallback]
        rows.append((mol, ";".join(s[:CFG["topk"]])))
    return pd.DataFrame(rows, columns=["molecule_id", "smiles"])

## Local validation (class-1 proxy)

Pick molecules from train, hold out roughly half their spectra as pseudo-test, and **remove those spectra from the library** (the other spectra of the same structure remain). Scoring uses `inchikey14` equality, same as the leaderboard's connectivity matching (without RDKit's tautomer canonicalisation). Restricting to `enveda-np-examples` / timsTOF gives the closest proxy to the real test. Set `N_VAL` small for speed.

In [ ]:
def mrr_at_k(ranked, truth, k=25):
    rr = []
    for mol, tk in truth.items():
        keys = [x for x, _ in ranked.get(mol, [])][:k]
        rr.append(1.0 / (keys.index(tk) + 1) if tk in keys else 0.0)
    return float(np.mean(rr)), rr

RUN_VAL = True
N_VAL = int(os.environ.get("N_VAL", 100))
if RUN_VAL:
    rng = np.random.RandomState(0)
    cnt = lib_meta.groupby("inchikey14").size()
    multi = cnt[cnt >= 2].index.to_numpy()
    val_keys = rng.choice(multi, size=min(N_VAL, len(multi)), replace=False)
    sel = lib_meta[lib_meta.inchikey14.isin(val_keys)]
    held = sel.groupby("inchikey14", group_keys=False).apply(lambda g: g.sample(frac=0.5, random_state=0) if len(g) > 2 else g.iloc[:1])
    held_ids = set(held.spectrum_id)
    tr = pq.ParquetFile(f"{DATA_DIR}/train.parquet")
    cols = ["spectrum_id", "inchikey14", "ms2_mzs", "ms2_normalized_intensities", "adduct", "precursor_mz"]
    parts = []
    for b in tr.iter_batches(batch_size=50000, columns=cols):
        t = pa.Table.from_batches([b]); t = t.filter(pc.is_in(t["spectrum_id"], value_set=pa.array(sorted(held_ids))))
        if t.num_rows: parts.append(t.to_pandas())
    val_df = pd.concat(parts, ignore_index=True)
    val_df = val_df[val_df.adduct.isin(ADDUCT_DELTA)].rename(columns={"inchikey14": "truth"})
    val_df["molecule_id"] = val_df["truth"]            # molecule = structure
    truth = val_df.drop_duplicates("molecule_id").set_index("molecule_id")["truth"].to_dict()
    ranked = predict(val_df, exclude_spectra=held_ids)
    score, rr = mrr_at_k(ranked, truth)
    print(f"validation MRR@25 = {score:.4f} on {len(truth)} molecules; top1 = {np.mean([r==1 for r in rr]):.3f}")

## Predict test and write `submission.csv`

In [ ]:
test = pq.read_table(f"{DATA_DIR}/test.parquet").to_pandas()
print(test.shape, test.molecule_id.nunique(), "molecules")
ranked = predict(test)
sub = to_submission(test, ranked)
n_empty = sum(1 for m in test.molecule_id.unique() if not ranked.get(m))
print(f"{n_empty} molecules without any library candidate (fallback SMILES used)")
sub.to_csv(f"{OUT_DIR}/submission.csv", index=False)
sub.head()

In [ ]:
# ---- submission validator ----
ss = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
chk = pd.read_csv(f"{OUT_DIR}/submission.csv")
assert list(chk.columns) == ["molecule_id", "smiles"]
assert chk.molecule_id.is_unique and chk.notna().all().all() and len(chk) > 0
assert set(chk.molecule_id) == set(ss.molecule_id), "molecule_id mismatch with sample_submission"
assert chk.smiles.str.count(";").add(1).max() <= 25
print("submission OK", chk.shape)